# PicoClimate Test EDA

Detailed EDA with derivative and shapelet features.
Update the configuration cell if needed.

In [1]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path("D:/repositories/personal/xai-spatio-temporal/data/picoclimate_test")
FIG_DIR = Path("D:/repositories/personal/xai-spatio-temporal/scripts/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = None  # optional: set to a specific CSV path
TIME_COL = None   # optional: set to known time column
GROUP_COLS = []   # optional: set to group keys, e.g., ["station_id"]
TARGET_COL = None # optional: exclude target from features/plots

PLOT_SAMPLE = 20000
MAX_NUM_COLS = 12

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 11,
})

def timestamp():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def save_fig(fig, name):
    out_path = FIG_DIR / f"{name}_{timestamp()}.png"
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    print(f"Saved: {out_path}")
    plt.close(fig)

def infer_time_col(columns):
    keys = ["time", "date", "timestamp"]
    for col in columns:
        lower = col.lower()
        if any(k in lower for k in keys):
            return col
    return None

def infer_group_cols(columns):
    keys = ["id", "station", "sensor", "device", "site"]
    return [col for col in columns if any(k in col.lower() for k in keys)]

In [2]:
# Source - https://stackoverflow.com/a/49189503
# Posted by YOLO, modified by community. See post 'Timeline' for change history
# Retrieved 2026-05-22, License - CC BY-SA 4.0

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)


In [3]:
csv_files = sorted(DATA_DIR.rglob("*.csv"))
print(f"Found {len(csv_files)} CSV files")
for f in csv_files:
    print(f)

if DATA_FILE is None:
    preferred_names = {"raw_measurements.csv", "window_features.csv"}
    preferred = [f for f in csv_files if f.name.lower() in preferred_names]
    DATA_FILE = preferred[0] if preferred else (csv_files[0] if csv_files else None)

print("Using:", DATA_FILE)

Found 2 CSV files
D:\repositories\personal\xai-spatio-temporal\data\picoclimate_test\tracks_measurements.csv
D:\repositories\personal\xai-spatio-temporal\data\picoclimate_test\window_features.csv
Using: D:\repositories\personal\xai-spatio-temporal\data\picoclimate_test\window_features.csv


In [4]:
if DATA_FILE is None:
    raise FileNotFoundError("No CSV file found in DATA_DIR.")

df = pd.read_csv(DATA_FILE)
print("Shape:", df.shape)
display(df.head())

Shape: (165, 100)


,track_id,date,city,time_slot,loc_index__slot_1,air_temp_c__slot_1,rel_humidity_pct__slot_1,wind_speed_ms__slot_1,wind_dir_deg__slot_1,pressure_hpa__slot_1,precipitation_mm__slot_1,solar_wm2__slot_1,longwave_wm2__slot_1,surface_temp_c__slot_1,soil_moisture_pct__slot_1,ndvi__slot_1,pm25_ugm3__slot_1,pm10_ugm3__slot_1,co2_ppm__slot_1,no2_ppb__slot_1,o3_ppb__slot_1,noise_db__slot_1,traffic_index__slot_1,pedestrian_index__slot_1,sky_view_factor__slot_1,impervious_fraction__slot_1,water_proximity__slot_1,heat_index_c__slot_1,loc_index__slot_2,air_temp_c__slot_2,rel_humidity_pct__slot_2,wind_speed_ms__slot_2,wind_dir_deg__slot_2,pressure_hpa__slot_2,precipitation_mm__slot_2,solar_wm2__slot_2,longwave_wm2__slot_2,surface_temp_c__slot_2,soil_moisture_pct__slot_2,ndvi__slot_2,pm25_ugm3__slot_2,pm10_ugm3__slot_2,co2_ppm__slot_2,no2_ppb__slot_2,o3_ppb__slot_2,noise_db__slot_2,traffic_index__slot_2,pedestrian_index__slot_2,sky_view_factor__slot_2,impervious_fraction__slot_2,water_proximity__slot_2,heat_index_c__slot_2,loc_index__slot_3,air_temp_c__slot_3,rel_humidity_pct__slot_3,wind_speed_ms__slot_3,wind_dir_deg__slot_3,pressure_hpa__slot_3,precipitation_mm__slot_3,solar_wm2__slot_3,longwave_wm2__slot_3,surface_temp_c__slot_3,soil_moisture_pct__slot_3,ndvi__slot_3,pm25_ugm3__slot_3,pm10_ugm3__slot_3,co2_ppm__slot_3,no2_ppb__slot_3,o3_ppb__slot_3,noise_db__slot_3,traffic_index__slot_3,pedestrian_index__slot_3,sky_view_factor__slot_3,impervious_fraction__slot_3,water_proximity__slot_3,heat_index_c__slot_3,loc_index__slot_4,air_temp_c__slot_4,rel_humidity_pct__slot_4,wind_speed_ms__slot_4,wind_dir_deg__slot_4,pressure_hpa__slot_4,precipitation_mm__slot_4,solar_wm2__slot_4,longwave_wm2__slot_4,surface_temp_c__slot_4,soil_moisture_pct__slot_4,ndvi__slot_4,pm25_ugm3__slot_4,pm10_ugm3__slot_4,co2_ppm__slot_4,no2_ppb__slot_4,o3_ppb__slot_4,noise_db__slot_4,traffic_index__slot_4,pedestrian_index__slot_4,sky_view_factor__slot_4,impervious_fraction__slot_4,water_proximity__slot_4,heat_index_c__slot_4
0,mon_track_000,2026-05-04,Montpellier,afternoon,99.470270,22.096012,59.305914,6.615490,179.382352,1012.053729,0.167463,337.875510,352.050019,22.077778,31.018985,0.465231,15.023713,24.724887,420.109991,25.056283,60.249485,54.669885,0.407717,0.312035,0.505851,0.603325,0.299239,23.026603,98.266304,21.921181,59.054201,6.597734,178.182975,1012.139217,0.283692,330.532703,350.704184,21.878758,30.407015,0.450565,14.834514,24.489383,419.368815,24.736515,59.251078,54.665393,0.397660,0.297516,0.500779,0.598014,0.297039,22.826601,98.854839,22.253119,58.766570,6.625406,180.502599,1012.084266,0.288945,336.957026,349.148887,21.941046,30.933251,0.464929,14.928497,25.028414,420.882446,25.471718,60.953890,55.136302,0.413146,0.316589,0.507291,0.599297,0.298116,23.129776,100.664773,22.006039,58.690345,6.617048,179.879620,1012.092673,0.194590,330.049333,347.784049,21.873380,31.157010,0.453211,15.070052,24.552696,418.408654,24.586340,59.573935,54.734635,0.396776,0.298410,0.495858,0.600226,0.312011,22.875073
1,mon_track_000,2026-05-05,Montpellier,noon,98.284153,21.992270,58.591666,6.685586,178.756134,1011.843593,0.266671,336.031262,349.536665,22.005822,30.842155,0.457580,14.792456,24.595063,419.822761,24.620148,60.340023,55.147079,0.397799,0.300698,0.503804,0.600164,0.300807,22.851437,99.957219,21.851682,58.877884,6.396632,178.976323,1012.116979,0.234172,326.662496,348.685410,21.748146,30.573307,0.451008,15.003223,24.710312,418.140267,24.728992,58.827607,54.747870,0.387706,0.288842,0.498743,0.590298,0.296408,22.739470,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,mon_track_000,2026-05-08,Montpellier,noon,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,99.564767,22.224048,58.981225,6.630385,179.621964,1011.990231,0.236260,335.739698,347.617298,22.058038,30.797912,0.458827,14.892932,

## Pipeline note: track-based sequences, alignment, and window features

The picoclimate fixture is track-based. If `loc_id` reflects visit order within a track, it can be treated as a pseudo-time slot axis; otherwise, the real date/time columns should define the ordering.

For mixed-length sequences, the alignment helpers in `scripts/alignment.txt` can be used to extend endpoints and resample each variable to a common length before shapelet extraction.

`window_features.csv` remains useful as a fixed-width representation for reproducible clustering baselines, quick EDA, and slide-ready summaries, even if the shapelet pipeline starts from the raw track sequences.


In [5]:
display(df.sample(min(5, len(df)), random_state=42))
df.info()
missing = df.isna().mean().sort_values(ascending=False)
print("Top missing columns:")
display(missing.head(20))
print("Duplicate rows:", df.duplicated().sum())
display(df.describe(include="all").T)

,track_id,date,city,time_slot,loc_index__slot_1,air_temp_c__slot_1,rel_humidity_pct__slot_1,wind_speed_ms__slot_1,wind_dir_deg__slot_1,pressure_hpa__slot_1,precipitation_mm__slot_1,solar_wm2__slot_1,longwave_wm2__slot_1,surface_temp_c__slot_1,soil_moisture_pct__slot_1,ndvi__slot_1,pm25_ugm3__slot_1,pm10_ugm3__slot_1,co2_ppm__slot_1,no2_ppb__slot_1,o3_ppb__slot_1,noise_db__slot_1,traffic_index__slot_1,pedestrian_index__slot_1,sky_view_factor__slot_1,impervious_fraction__slot_1,water_proximity__slot_1,heat_index_c__slot_1,loc_index__slot_2,air_temp_c__slot_2,rel_humidity_pct__slot_2,wind_speed_ms__slot_2,wind_dir_deg__slot_2,pressure_hpa__slot_2,precipitation_mm__slot_2,solar_wm2__slot_2,longwave_wm2__slot_2,surface_temp_c__slot_2,soil_moisture_pct__slot_2,ndvi__slot_2,pm25_ugm3__slot_2,pm10_ugm3__slot_2,co2_ppm__slot_2,no2_ppb__slot_2,o3_ppb__slot_2,noise_db__slot_2,traffic_index__slot_2,pedestrian_index__slot_2,sky_view_factor__slot_2,impervious_fraction__slot_2,water_proximity__slot_2,heat_index_c__slot_2,loc_index__slot_3,air_temp_c__slot_3,rel_humidity_pct__slot_3,wind_speed_ms__slot_3,wind_dir_deg__slot_3,pressure_hpa__slot_3,precipitation_mm__slot_3,solar_wm2__slot_3,longwave_wm2__slot_3,surface_temp_c__slot_3,soil_moisture_pct__slot_3,ndvi__slot_3,pm25_ugm3__slot_3,pm10_ugm3__slot_3,co2_ppm__slot_3,no2_ppb__slot_3,o3_ppb__slot_3,noise_db__slot_3,traffic_index__slot_3,pedestrian_index__slot_3,sky_view_factor__slot_3,impervious_fraction__slot_3,water_proximity__slot_3,heat_index_c__slot_3,loc_index__slot_4,air_temp_c__slot_4,rel_humidity_pct__slot_4,wind_speed_ms__slot_4,wind_dir_deg__slot_4,pressure_hpa__slot_4,precipitation_mm__slot_4,solar_wm2__slot_4,longwave_wm2__slot_4,surface_temp_c__slot_4,soil_moisture_pct__slot_4,ndvi__slot_4,pm25_ugm3__slot_4,pm10_ugm3__slot_4,co2_ppm__slot_4,no2_ppb__slot_4,o3_ppb__slot_4,noise_db__slot_4,traffic_index__slot_4,pedestrian_index__slot_4,sky_view_factor__slot_4,impervious_fraction__slot_4,water_proximity__slot_4,heat_index_c__slot_4
135,nan_track_007,2026-05-05,Nantes,morning,70.117188,18.720776,71.083109,5.523825,181.510734,1011.909400,0.135432,281.200067,353.014610,21.887745,39.505761,0.547773,15.306189,25.135498,419.901619,25.413276,60.378687,55.090040,0.392447,0.303349,0.503055,0.597829,0.294248,20.829087,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
115,nan_track_005,2026-05-15,Nantes,afternoon,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,82.532051,18.889458,70.959382,5.632726,176.895727,1012.305789,0.278191,279.065928,350.646770,21.920319,39.526241,0.549752,15.122847,24.531717,420.220111,24.783154,60.177635,54.902480,0.418068,0.300142,0.500202,0.596276,0.304733,20.985397,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
131,nan_track_006,2026-05-20,Nantes,afternoon,54.165049,18.837213,71.345930,5.598903,179.051710,1012.222811,0.179166,273.020075,348.212376,21.985485,39.995809,0.535247,15.339535,25.024084,419.772792,25.219889,59.771052,55.083958,0.386445,0.296679,0.492984,0.597130,0.314249,20.971806,55.233010,19.084502,70.846668,5.631470,183.995727,1011.771118,0.311820,277.161690,351.012659,21.906639,39.579309,0.538255,14.670056,24.929522,418.190865,24.873621,60.536912,54.940606,0.384554,0.289669,0.501801,0.602102,0.302139,21.169169,54.895238,18.970585,70.517798,5.597959,181.461544,1011.949112,0.207709,276.278845,346.546230,21.869013,40.069940,0.539286,15.187683,25.033975,418.928854,24.655306,59.008462,54.710884,0.411582,0.296840,0.497121,0.599381,0.305617,21.022365,54.657143,18.994663,70.871763,5.644504,179.496383,1011.821812,0.257457,283.791698,34

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 165 entries, 0 to 164
Data columns (total 100 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   track_id                     165 non-null    object 
 1   date                         165 non-null    object 
 2   city                         165 non-null    object 
 3   time_slot                    165 non-null    object 
 4   loc_index__slot_1            93 non-null     float64
 5   air_temp_c__slot_1           93 non-null     float64
 6   rel_humidity_pct__slot_1     93 non-null     float64
 7   wind_speed_ms__slot_1        93 non-null     float64
 8   wind_dir_deg__slot_1         93 non-null     float64
 9   pressure_hpa__slot_1         93 non-null     float64
 10  precipitation_mm__slot_1     93 non-null     float64
 11  solar_wm2__slot_1            93 non-null     float64
 12  longwave_wm2__slot_1         93 non-null     float64
 13  surface_temp_c__slo

wind_speed_ms__slot_1        0.436364
rel_humidity_pct__slot_1     0.436364
air_temp_c__slot_1           0.436364
loc_index__slot_1            0.436364
o3_ppb__slot_1               0.436364
noise_db__slot_1             0.436364
traffic_index__slot_1        0.436364
pedestrian_index__slot_1     0.436364
wind_dir_deg__slot_1         0.436364
pressure_hpa__slot_1         0.436364
precipitation_mm__slot_1     0.436364
solar_wm2__slot_1            0.436364
longwave_wm2__slot_1         0.436364
surface_temp_c__slot_1       0.436364
soil_moisture_pct__slot_1    0.436364
ndvi__slot_1                 0.436364
pm25_ugm3__slot_1            0.436364
pm10_ugm3__slot_1            0.436364
co2_ppm__slot_1              0.436364
no2_ppb__slot_1              0.436364
dtype: float64

Duplicate rows: 0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
track_id,165,16,mon_track_004,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date,165,21,2026-05-18,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,165,2,Nantes,110,NaN,NaN,NaN,NaN,NaN,NaN,NaN
time_slot,165,4,afternoon,47,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loc_index__slot_1,93.0,NaN,NaN,NaN,72.744461,14.156981,53.528302,63.586777,69.8,82.703226,99.47027
air_temp_c__slot_1,93.0,NaN,NaN,NaN,19.956103,1.41804,18.646585,18.929452,19.088582,21.840958,22.278654
rel_humidity_pct__slot_1,93.0,NaN,NaN,NaN,67.167855,5.592607,57.628797,59.623748,70.702946,71.09487,71.844638
wind_speed_ms__slot_1,93.0,NaN,NaN,NaN,5.93088,0.469794,5.427177,5.582558,5.662686,6.519218,6.769181
wind_dir_deg__slot_1,93.0,NaN,NaN,NaN,179.953489,2.793273,173.29016,178.506646,180.265981,181.510734,189.006152
pressure_hpa__slot_1,93.0,NaN,NaN,NaN,1012.015054,0.205076,1011.405855,1011.868213,1012.004367,1012.166042,1012.608141


In [6]:
if TIME_COL is None:
    TIME_COL = infer_time_col(df.columns)
if not GROUP_COLS:
    GROUP_COLS = infer_group_cols(df.columns)

print("TIME_COL:", TIME_COL)
print("GROUP_COLS:", GROUP_COLS)

if TIME_COL:
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    sort_cols = GROUP_COLS + [TIME_COL] if GROUP_COLS else [TIME_COL]
    df = df.sort_values(sort_cols)

TIME_COL: date
GROUP_COLS: ['track_id', 'rel_humidity_pct__slot_1', 'rel_humidity_pct__slot_2', 'rel_humidity_pct__slot_3', 'rel_humidity_pct__slot_4']


In [7]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
if TARGET_COL in numeric_cols:
    numeric_cols.remove(TARGET_COL)
print("Numeric columns:", numeric_cols)

Numeric columns: ['loc_index__slot_1', 'air_temp_c__slot_1', 'rel_humidity_pct__slot_1', 'wind_speed_ms__slot_1', 'wind_dir_deg__slot_1', 'pressure_hpa__slot_1', 'precipitation_mm__slot_1', 'solar_wm2__slot_1', 'longwave_wm2__slot_1', 'surface_temp_c__slot_1', 'soil_moisture_pct__slot_1', 'ndvi__slot_1', 'pm25_ugm3__slot_1', 'pm10_ugm3__slot_1', 'co2_ppm__slot_1', 'no2_ppb__slot_1', 'o3_ppb__slot_1', 'noise_db__slot_1', 'traffic_index__slot_1', 'pedestrian_index__slot_1', 'sky_view_factor__slot_1', 'impervious_fraction__slot_1', 'water_proximity__slot_1', 'heat_index_c__slot_1', 'loc_index__slot_2', 'air_temp_c__slot_2', 'rel_humidity_pct__slot_2', 'wind_speed_ms__slot_2', 'wind_dir_deg__slot_2', 'pressure_hpa__slot_2', 'precipitation_mm__slot_2', 'solar_wm2__slot_2', 'longwave_wm2__slot_2', 'surface_temp_c__slot_2', 'soil_moisture_pct__slot_2', 'ndvi__slot_2', 'pm25_ugm3__slot_2', 'pm10_ugm3__slot_2', 'co2_ppm__slot_2', 'no2_ppb__slot_2', 'o3_ppb__slot_2', 'noise_db__slot_2', 'traffic

In [8]:
if numeric_cols:
    missing_pct = df[numeric_cols].isna().mean().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(12, 6))
    missing_pct.head(30).plot(kind="bar", ax=ax)
    ax.set_title("Missingness (top 30 numeric columns)")
    ax.set_ylabel("Fraction missing")
    save_fig(fig, "missingness_numeric")

Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\missingness_numeric_20260529_062150.png


In [9]:
if numeric_cols:
    cols = numeric_cols[:MAX_NUM_COLS]
    plot_df = df[cols]
    if len(plot_df) > PLOT_SAMPLE:
        plot_df = plot_df.sample(PLOT_SAMPLE, random_state=42)

    ncols = 3
    nrows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(14, 4 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, cols):
        sns.histplot(plot_df[col].dropna(), bins=40, ax=ax, kde=True)
        ax.set_title(col)
    for ax in axes[len(cols):]:
        ax.axis("off")
    save_fig(fig, "numeric_hist")

    fig, ax = plt.subplots(figsize=(12, 6))
    sns.boxplot(data=plot_df, orient="h", ax=ax)
    ax.set_title("Numeric column boxplots (sampled)")
    save_fig(fig, "numeric_boxplot")

Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\numeric_hist_20260529_062154.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\numeric_boxplot_20260529_062157.png


In [10]:
if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr()
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(corr, cmap="vlag", center=0, ax=ax)
    ax.set_title("Correlation heatmap")
    save_fig(fig, "correlation_heatmap")

Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\correlation_heatmap_20260529_062158.png


In [11]:
if TIME_COL and numeric_cols:
    sample_cols = numeric_cols[: min(4, len(numeric_cols))]
    if GROUP_COLS:
        first_key = df[GROUP_COLS].drop_duplicates().iloc[0].to_list()
        mask = np.ones(len(df), dtype=bool)
        for col, val in zip(GROUP_COLS, first_key):
            mask &= df[col] == val
        ts_df = df.loc[mask, [TIME_COL] + sample_cols].sort_values(TIME_COL)
        title_suffix = " (first group)"
    else:
        ts_df = df[[TIME_COL] + sample_cols].sort_values(TIME_COL)
        title_suffix = ""

    if len(ts_df) > PLOT_SAMPLE:
        ts_df = ts_df.iloc[:PLOT_SAMPLE]

    fig, ax = plt.subplots(figsize=(12, 6))
    for col in sample_cols:
        ax.plot(ts_df[TIME_COL], ts_df[col], label=col)
    ax.set_title(f"Time series sample{title_suffix}")
    ax.set_xlabel(TIME_COL)
    ax.legend()
    save_fig(fig, "timeseries_sample")

Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\timeseries_sample_20260529_062200.png


## Additional EDA
More distribution, missingness, outliers, pairplot, and rolling stats.

In [12]:
EXTRA_SAMPLE = 5000
PAIRPLOT_SAMPLE = 2000
MAX_CAT_COLS = 6
CAT_TOP_N = 12
MISSINGNESS_COLS = 30
ROLLING_WINDOW = 24

def safe_name(name):
    return "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in str(name))[:80]

row_count, col_count = df.shape
print(f"Rows: {row_count}, Columns: {col_count}")
print(f"Numeric cols: {len(numeric_cols)}")

cat_cols = [c for c in df.columns if c not in numeric_cols and c != TIME_COL]
if TARGET_COL in cat_cols:
    cat_cols.remove(TARGET_COL)
print("Categorical cols:", cat_cols)

if cat_cols:
    cardinality = df[cat_cols].nunique(dropna=True).sort_values(ascending=False)
    display(cardinality.head(20))

if GROUP_COLS:
    group_sizes = df.groupby(GROUP_COLS, dropna=False).size().sort_values(ascending=False)
    display(group_sizes.head(20))
    fig, ax = plt.subplots(figsize=(12, 4))
    group_sizes.head(20).plot(kind="bar", ax=ax)
    ax.set_title("Top group sizes")
    save_fig(fig, "group_sizes")

for col in cat_cols[:MAX_CAT_COLS]:
    vc = df[col].astype("string").value_counts(dropna=False).head(CAT_TOP_N)
    fig, ax = plt.subplots(figsize=(10, 4))
    vc.sort_values().plot(kind="barh", ax=ax)
    ax.set_title(f"Top categories: {col}")
    save_fig(fig, f"cat_{safe_name(col)}")

if TARGET_COL and TARGET_COL in df.columns:
    if pd.api.types.is_numeric_dtype(df[TARGET_COL]):
        fig, ax = plt.subplots(figsize=(10, 5))
        sns.histplot(df[TARGET_COL].dropna(), bins=40, ax=ax)
        ax.set_title(f"Target distribution: {TARGET_COL}")
        save_fig(fig, f"target_{safe_name(TARGET_COL)}")
    else:
        vc = df[TARGET_COL].astype("string").value_counts(dropna=False).head(CAT_TOP_N)
        fig, ax = plt.subplots(figsize=(10, 4))
        vc.sort_values().plot(kind="barh", ax=ax)
        ax.set_title(f"Target distribution: {TARGET_COL}")
        save_fig(fig, f"target_{safe_name(TARGET_COL)}")

if numeric_cols:
    miss_cols = df[numeric_cols].isna().mean().sort_values(ascending=False).head(MISSINGNESS_COLS).index
    miss_df = df[miss_cols]
    if len(miss_df) > EXTRA_SAMPLE:
        miss_df = miss_df.sample(EXTRA_SAMPLE, random_state=42)
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.heatmap(miss_df.isna(), cbar=False, ax=ax)
    ax.set_title("Missingness heatmap (sampled)")
    save_fig(fig, "missingness_heatmap")

if numeric_cols:
    stats = pd.DataFrame({
        "mean": df[numeric_cols].mean(),
        "std": df[numeric_cols].std(),
        "skew": df[numeric_cols].skew(),
        "kurtosis": df[numeric_cols].kurtosis(),
    }).sort_values("skew", key=lambda s: s.abs(), ascending=False)
    display(stats.head(20))

    def outlier_rate(s):
        s = s.dropna()
        if len(s) < 4:
            return np.nan
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            return 0.0
        low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        return ((s < low) | (s > high)).mean()

    out_rates = df[numeric_cols].apply(outlier_rate).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(12, 6))
    out_rates.head(30).plot(kind="bar", ax=ax)
    ax.set_title("Outlier rate (IQR) - top 30")
    ax.set_ylabel("Fraction outliers")
    save_fig(fig, "outlier_rate")

if numeric_cols:
    pair_cols = numeric_cols[: min(5, len(numeric_cols))]
    pair_df = df[pair_cols].dropna()
    if len(pair_df) > PAIRPLOT_SAMPLE:
        pair_df = pair_df.sample(PAIRPLOT_SAMPLE, random_state=42)
    if len(pair_df) > 1:
        g = sns.pairplot(pair_df, corner=True, diag_kind="hist", plot_kws={"s": 12, "alpha": 0.4})
        save_fig(g.fig, "pairplot_sample")

if TIME_COL and numeric_cols:
    from pandas.plotting import autocorrelation_plot
    col = numeric_cols[0]
    ts_df = df[[TIME_COL, col]].dropna().sort_values(TIME_COL)
    if len(ts_df) > PLOT_SAMPLE:
        ts_df = ts_df.iloc[:PLOT_SAMPLE]
    if len(ts_df) > 1:
        fig, ax = plt.subplots(figsize=(10, 4))
        autocorrelation_plot(ts_df[col], ax=ax)
        ax.set_title(f"Autocorrelation: {col}")
        save_fig(fig, f"autocorr_{safe_name(col)}")

        roll = ts_df[col].rolling(ROLLING_WINDOW, min_periods=max(2, ROLLING_WINDOW // 4))
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.plot(ts_df[TIME_COL], ts_df[col], alpha=0.35, label="value")
        ax.plot(ts_df[TIME_COL], roll.mean(), label=f"rolling mean ({ROLLING_WINDOW})")
        ax.plot(ts_df[TIME_COL], roll.std(), label=f"rolling std ({ROLLING_WINDOW})")
        ax.set_title(f"Rolling stats: {col}")
        ax.legend()
        save_fig(fig, f"rolling_stats_{safe_name(col)}")

Rows: 165, Columns: 100
Numeric cols: 96
Categorical cols: ['track_id', 'city', 'time_slot']


track_id     16
time_slot     4
city          2
dtype: int64

track_id       rel_humidity_pct__slot_1  rel_humidity_pct__slot_2  rel_humidity_pct__slot_3  rel_humidity_pct__slot_4
mon_track_000  58.591666                 58.877884                 NaN                       NaN                         1
               58.970603                 58.798039                 58.673252                 59.120539                   1
               59.045796                 58.773119                 NaN                       59.257902                   1
               59.305914                 59.054201                 58.766570                 58.690345                   1
               NaN                       58.981225                 59.499280                 NaN                         1
                                         59.005125                 58.275184                 58.119394                   1
                                         59.268233                 58.567172                 59.510350                   1
mon_track_001  58.948

C:\Users\Admin\AppData\Local\Temp\ipykernel_4832\501949634.py:33: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()


Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\group_sizes_20260529_062201.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\cat_track_id_20260529_062201.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\cat_city_20260529_062202.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\cat_time_slot_20260529_062202.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\missingness_heatmap_20260529_062203.png


,mean,std,skew,kurtosis
rel_humidity_pct__slot_3,67.888302,5.239126,-1.142713,-0.664748
ndvi__slot_3,0.518630,0.035000,-1.133148,-0.542757
wind_speed_ms__slot_3,5.847090,0.453576,1.133041,-0.505295
soil_moisture_pct__slot_3,37.790147,3.918383,-1.130984,-0.664394
air_temp_c__slot_3,19.735367,1.320545,1.126130,-0.641575
heat_index_c__slot_3,21.524197,0.812593,1.058221,-0.566391
solar_wm2__slot_3,293.487545,25.192053,1.051044,-0.537814
rel_humidity_pct__slot_1,67.167855,5.592607,-0.768956,-1.402085
soil_moisture_pct__slot_1,37.035655,4.224914,-0.762772,-1.409053
air_temp_c__slot_1,19.956103,1.418040,0.761313,-1.393390


Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\outlier_rate_20260529_062204.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\pairplot_sample_20260529_062207.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\autocorr_loc_index__slot_1_20260529_062209.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\rolling_stats_loc_index__slot_1_20260529_062209.png


## Distribution report
Target distribution, feature distributions, log checks, and bivariate views.

In [13]:
DIST_MAX_NUM_COLS = 8
DIST_MAX_CAT_COLS = 6
TARGET_TOP_N = 20
BIVAR_MAX_NUM = 4
BIVAR_MAX_CAT = 4
BIVAR_SAMPLE = 5000
SKEW_THRESHOLD = 1.0

def is_classification_target(series):
    if pd.api.types.is_object_dtype(series) or pd.api.types.is_bool_dtype(series) or pd.api.types.is_categorical_dtype(series):
        return True
    return series.nunique(dropna=True) <= TARGET_TOP_N

def plot_target_distribution(df_in, target_col):
    if not target_col or target_col not in df_in.columns:
        print("TARGET_COL not set; skipping target plots.")
        return
    s = df_in[target_col]
    if is_classification_target(s):
        vc = s.astype("string").value_counts(dropna=False)
        total = vc.sum()
        top = vc.head(TARGET_TOP_N)
        fig, ax = plt.subplots(figsize=(10, 5))
        top.sort_values().plot(kind="barh", ax=ax)
        for p in ax.patches:
            pct = 100.0 * p.get_width() / total if total else 0.0
            ax.text(p.get_width() + 0.01 * total, p.get_y() + p.get_height() / 2, f"{pct:.1f}%")
        ax.set_title(f"Target distribution (classification): {target_col}")
        save_fig(fig, f"target_class_{safe_name(target_col)}")
    else:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        sns.histplot(s.dropna(), bins=40, kde=True, ax=axes[0])
        sns.boxplot(x=s.dropna(), ax=axes[1])
        axes[0].set_title(f"Target histogram: {target_col}")
        axes[1].set_title("Target boxplot")
        save_fig(fig, f"target_reg_{safe_name(target_col)}")

def numeric_summary_table(df_in, cols):
    if not cols:
        return
    summary = df_in[cols].describe().T
    summary["skew"] = df_in[cols].skew()
    summary["kurtosis"] = df_in[cols].kurtosis()
    display(summary.head(20))

def plot_numeric_distributions(df_in, cols, max_cols):
    for col in cols[:max_cols]:
        s = df_in[col].dropna()
        if s.empty:
            continue
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        sns.histplot(s, bins=40, kde=True, ax=axes[0])
        sns.boxplot(x=s, ax=axes[1])
        axes[0].set_title(f"{col} (skew={s.skew():.2f})")
        axes[1].set_title("Boxplot")
        save_fig(fig, f"dist_{safe_name(col)}")

def log1p_shift(series):
    s = series.dropna()
    if s.empty:
        return s, 0.0
    min_val = s.min()
    shift = -min_val + 1 if min_val <= -1 else 0.0
    return np.log1p(s + shift), shift

def plot_log_transform_comparison(df_in, cols, threshold):
    for col in cols:
        s = df_in[col].dropna()
        if s.empty:
            continue
        skew = s.skew()
        if abs(skew) < threshold:
            continue
        logged, shift = log1p_shift(s)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        sns.histplot(s, bins=40, kde=True, ax=axes[0])
        sns.histplot(logged, bins=40, kde=True, ax=axes[1])
        axes[0].set_title(f"{col} original")
        axes[1].set_title(f"log1p (shift={shift:.2f})")
        save_fig(fig, f"log_compare_{safe_name(col)}")

def plot_bivariate_with_target(df_in, target_col):
    if not target_col or target_col not in df_in.columns:
        return
    if is_classification_target(df_in[target_col]):
        for col in numeric_cols[:BIVAR_MAX_NUM]:
            if col == target_col:
                continue
            sample = df_in[[col, target_col]].dropna()
            if len(sample) > BIVAR_SAMPLE:
                sample = sample.sample(BIVAR_SAMPLE, random_state=42)
            if sample.empty:
                continue
            fig, ax = plt.subplots(figsize=(10, 5))
            sns.violinplot(data=sample, x=target_col, y=col, inner="quartile", cut=0, ax=ax)
            ax.set_title(f"{col} by {target_col}")
            ax.tick_params(axis="x", rotation=30)
            save_fig(fig, f"bivar_{safe_name(col)}_by_{safe_name(target_col)}")

        for col in cat_cols[:BIVAR_MAX_CAT]:
            ct = pd.crosstab(df_in[col].astype("string"), df_in[target_col].astype("string"), normalize="index")
            if ct.empty:
                continue
            top = df_in[col].astype("string").value_counts().head(CAT_TOP_N).index
            ct = ct.loc[ct.index.intersection(top)]
            fig, ax = plt.subplots(figsize=(10, 5))
            ct.plot(kind="bar", stacked=True, ax=ax)
            ax.set_title(f"{col} vs {target_col} (row %)")
            ax.legend(title=target_col, bbox_to_anchor=(1.02, 1), loc="upper left")
            save_fig(fig, f"bivar_{safe_name(col)}_stacked")
    else:
        for col in numeric_cols[:BIVAR_MAX_NUM]:
            if col == target_col:
                continue
            sample = df_in[[col, target_col]].dropna()
            if len(sample) > BIVAR_SAMPLE:
                sample = sample.sample(BIVAR_SAMPLE, random_state=42)
            if sample.empty:
                continue
            fig, ax = plt.subplots(figsize=(10, 5))
            sns.scatterplot(data=sample, x=col, y=target_col, s=10, alpha=0.4, ax=ax)
            ax.set_title(f"{target_col} vs {col}")
            save_fig(fig, f"bivar_{safe_name(target_col)}_vs_{safe_name(col)}")

        for col in cat_cols[:BIVAR_MAX_CAT]:
            sample = df_in[[col, target_col]].dropna()
            if sample.empty:
                continue
            top = sample[col].astype("string").value_counts().head(CAT_TOP_N).index
            sample = sample[sample[col].astype("string").isin(top)]
            fig, ax = plt.subplots(figsize=(12, 5))
            sns.boxplot(data=sample, x=col, y=target_col, ax=ax)
            ax.tick_params(axis="x", rotation=30)
            ax.set_title(f"{target_col} by {col}")
            save_fig(fig, f"bivar_{safe_name(target_col)}_by_{safe_name(col)}")

        if target_col in numeric_cols:
            corr = df_in[numeric_cols].corrwith(df_in[target_col]).dropna().sort_values()
            fig, ax = plt.subplots(figsize=(10, 6))
            corr.tail(20).plot(kind="barh", ax=ax)
            ax.set_title(f"Correlation with {target_col} (top 20)")
            save_fig(fig, f"corr_with_{safe_name(target_col)}")

if numeric_cols:
    row_missing = df[numeric_cols].isna().mean(axis=1)
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.histplot(row_missing, bins=40, ax=ax)
    ax.set_title("Missingness per row (numeric)")
    save_fig(fig, "missingness_row")

plot_target_distribution(df, TARGET_COL)
numeric_summary_table(df, numeric_cols)
plot_numeric_distributions(df, numeric_cols, DIST_MAX_NUM_COLS)
plot_log_transform_comparison(df, numeric_cols[:DIST_MAX_NUM_COLS], SKEW_THRESHOLD)
plot_bivariate_with_target(df, TARGET_COL)

Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\missingness_row_20260529_062210.png
TARGET_COL not set; skipping target plots.


,count,mean,std,min,25%,50%,75%,max,skew,kurtosis
loc_index__slot_1,93.0,72.744461,14.156981,53.528302,63.586777,69.800000,82.703226,99.470270,0.441486,-0.897301
air_temp_c__slot_1,93.0,19.956103,1.418040,18.646585,18.929452,19.088582,21.840958,22.278654,0.761313,-1.393390
rel_humidity_pct__slot_1,93.0,67.167855,5.592607,57.628797,59.623748,70.702946,71.094870,71.844638,-0.768956,-1.402085
wind_speed_ms__slot_1,93.0,5.930880,0.469794,5.427177,5.582558,5.662686,6.519218,6.769181,0.730843,-1.316687
wind_dir_deg__slot_1,93.0,179.953489,2.793273,173.290160,178.506646,180.265981,181.510734,189.006152,-0.022931,0.923537
pressure_hpa__slot_1,93.0,1012.015054,0.205076,1011.405855,1011.868213,1012.004367,1012.166042,1012.608141,0.175474,0.538823
precipitation_mm__slot_1,93.0,0.232184,0.052713,0.114196,0.206221,0.235958,0.261362,0.417823,0.168329,0.773974
solar_wm2__slot_1,93.0,297.395798,26.770976,266.883407,276.842163,284.112766,331.436026,350.675827,0.714566,-1.268137
longwave_wm2__slot_1,93.0,350.167094,2.596127,344.218393,348.303506,349.852463,352.050019,356.233784,0.010328,-0.236234
surface_temp_c__slot_1,93.0,21.968162,0.181294,21.452081,21.850028,21.967972,22.077778,22.366366,-0.066879,0.244612


Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\dist_loc_index__slot_1_20260529_062211.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\dist_air_temp_c__slot_1_20260529_062211.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\dist_rel_humidity_pct__slot_1_20260529_062212.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\dist_wind_speed_ms__slot_1_20260529_062213.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\dist_wind_dir_deg__slot_1_20260529_062213.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\dist_pressure_hpa__slot_1_20260529_062214.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\dist_precipitation_mm__slot_1_20260529_062215.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\dist_solar_wm2__slot_1_20260529_062216.png


In [14]:
DERIVATIVE_COLS = numeric_cols  # override if you want fewer columns

def add_derivatives(df_in, cols, group_cols, time_col):
    df_out = df_in.copy()
    if time_col:
        sort_cols = group_cols + [time_col] if group_cols else [time_col]
        df_out = df_out.sort_values(sort_cols)

    if group_cols:
        g = df_out.groupby(group_cols, sort=False, dropna=False)
        for col in cols:
            df_out[f"{col}__d1"] = g[col].diff()
            df_out[f"{col}__d2"] = df_out.groupby(group_cols, sort=False, dropna=False)[f"{col}__d1"].diff()
            df_out[f"{col}__d3"] = df_out.groupby(group_cols, sort=False, dropna=False)[f"{col}__d2"].diff()
    else:
        for col in cols:
            df_out[f"{col}__d1"] = df_out[col].diff()
            df_out[f"{col}__d2"] = df_out[f"{col}__d1"].diff()
            df_out[f"{col}__d3"] = df_out[f"{col}__d2"].diff()
    return df_out

df_deriv = add_derivatives(df, DERIVATIVE_COLS, GROUP_COLS, TIME_COL)
print("Derivative features added:", [c for c in df_deriv.columns if c.endswith("__d1")][:5])

C:\Users\Admin\AppData\Local\Temp\ipykernel_4832\1176366730.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_out[f"{col}__d2"] = df_out.groupby(group_cols, sort=False, dropna=False)[f"{col}__d1"].diff()
C:\Users\Admin\AppData\Local\Temp\ipykernel_4832\1176366730.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_out[f"{col}__d3"] = df_out.groupby(group_cols, sort=False, dropna=False)[f"{col}__d2"].diff()
C:\Users\Admin\AppData\Local\Temp\ipykernel_4832\1176366730.py:12: PerformanceWarning: DataFrame is highly fragmen

Derivative features added: ['loc_index__slot_1__d1', 'air_temp_c__slot_1__d1', 'rel_humidity_pct__slot_1__d1', 'wind_speed_ms__slot_1__d1', 'wind_dir_deg__slot_1__d1']


C:\Users\Admin\AppData\Local\Temp\ipykernel_4832\1176366730.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_out[f"{col}__d3"] = df_out.groupby(group_cols, sort=False, dropna=False)[f"{col}__d2"].diff()
C:\Users\Admin\AppData\Local\Temp\ipykernel_4832\1176366730.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_out[f"{col}__d1"] = g[col].diff()
C:\Users\Admin\AppData\Local\Temp\ipykernel_4832\1176366730.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` m

In [15]:
SHAPELET_SOURCE_COL = None  # set to a numeric column for shapelets
SHAPELET_LENGTH = 30
N_SHAPELETS = 6
MAX_GROUPS = 50
MAX_POINTS_PER_SERIES = 2000
RANDOM_SEED = 42

def z_normalize(x, eps=1e-8):
    x = np.asarray(x, dtype=float)
    m = np.nanmean(x)
    s = np.nanstd(x)
    if s < eps:
        return x * 0.0
    return (x - m) / s

def min_distance_to_shapelet(series, shapelet):
    if len(series) < len(shapelet):
        return np.nan
    series = z_normalize(series)
    try:
        windows = np.lib.stride_tricks.sliding_window_view(series, len(shapelet))
    except AttributeError:
        windows = np.array([series[i : i + len(shapelet)] for i in range(len(series) - len(shapelet) + 1)])
    dists = np.linalg.norm(windows - shapelet, axis=1)
    return float(np.min(dists))

shapelet_df = None
if numeric_cols:
    if SHAPELET_SOURCE_COL is None:
        SHAPELET_SOURCE_COL = numeric_cols[0]

    rng = np.random.default_rng(RANDOM_SEED)
    series_list = []
    group_keys = []

    if GROUP_COLS:
        groups = df_deriv.groupby(GROUP_COLS, sort=False, dropna=False)
        keys = list(groups.groups.keys())[:MAX_GROUPS]
        for key in keys:
            if isinstance(key, tuple):
                if any(pd.isna(v) for v in key):
                    continue
                mask = np.ones(len(df_deriv), dtype=bool)
                for col, val in zip(GROUP_COLS, key):
                    mask &= df_deriv[col].eq(val)
                gdf = df_deriv.loc[mask]
            else:
                if pd.isna(key):
                    continue
                gdf = df_deriv[df_deriv[GROUP_COLS[0]].eq(key)]
            series = gdf[SHAPELET_SOURCE_COL].to_numpy()
            series = pd.Series(series).interpolate(limit_direction="both").to_numpy()
            series = series[:MAX_POINTS_PER_SERIES]
            if len(series) >= SHAPELET_LENGTH:
                series_list.append(series)
                group_keys.append(key)
    else:
        series = df_deriv[SHAPELET_SOURCE_COL].to_numpy()
        series = pd.Series(series).interpolate(limit_direction="both").to_numpy()
        series = series[:MAX_POINTS_PER_SERIES]
        if len(series) >= SHAPELET_LENGTH:
            series_list.append(series)
            group_keys.append(None)

    if series_list:
        shapelets = []
        for _ in range(N_SHAPELETS):
            s = series_list[rng.integers(len(series_list))]
            start = rng.integers(0, len(s) - SHAPELET_LENGTH + 1)
            shapelets.append(z_normalize(s[start : start + SHAPELET_LENGTH]))

        feature_rows = []
        for series, key in zip(series_list, group_keys):
            feats = [min_distance_to_shapelet(series, shp) for shp in shapelets]
            row = {f"shapelet_{i+1}_dist": feats[i] for i in range(len(feats))}
            if GROUP_COLS:
                if isinstance(key, tuple):
                    for col, val in zip(GROUP_COLS, key):
                        row[col] = val
                else:
                    row[GROUP_COLS[0]] = key
            feature_rows.append(row)

        shapelet_df = pd.DataFrame(feature_rows)

        fig, ax = plt.subplots(figsize=(10, 5))
        for i, shp in enumerate(shapelets):
            ax.plot(shp, label=f"shapelet_{i+1}")
        ax.set_title("Sample shapelets")
        ax.legend()
        save_fig(fig, "shapelets_sample")

In [16]:
df_feat = df_deriv.copy()
if shapelet_df is not None:
    if GROUP_COLS:
        df_feat = df_feat.merge(shapelet_df, on=GROUP_COLS, how="left")
    else:
        for col in shapelet_df.columns:
            df_feat[col] = shapelet_df[col].iloc[0]

print("Final shape:", df_feat.shape)
display(df_feat.head())

Final shape: (165, 388)


,track_id,date,city,time_slot,loc_index__slot_1,air_temp_c__slot_1,rel_humidity_pct__slot_1,wind_speed_ms__slot_1,wind_dir_deg__slot_1,pressure_hpa__slot_1,precipitation_mm__slot_1,solar_wm2__slot_1,longwave_wm2__slot_1,surface_temp_c__slot_1,soil_moisture_pct__slot_1,ndvi__slot_1,pm25_ugm3__slot_1,pm10_ugm3__slot_1,co2_ppm__slot_1,no2_ppb__slot_1,o3_ppb__slot_1,noise_db__slot_1,traffic_index__slot_1,pedestrian_index__slot_1,sky_view_factor__slot_1,impervious_fraction__slot_1,water_proximity__slot_1,heat_index_c__slot_1,loc_index__slot_2,air_temp_c__slot_2,rel_humidity_pct__slot_2,wind_speed_ms__slot_2,wind_dir_deg__slot_2,pressure_hpa__slot_2,precipitation_mm__slot_2,solar_wm2__slot_2,longwave_wm2__slot_2,surface_temp_c__slot_2,soil_moisture_pct__slot_2,ndvi__slot_2,pm25_ugm3__slot_2,pm10_ugm3__slot_2,co2_ppm__slot_2,no2_ppb__slot_2,o3_ppb__slot_2,noise_db__slot_2,traffic_index__slot_2,pedestrian_index__slot_2,sky_view_factor__slot_2,impervious_fraction__slot_2,water_proximity__slot_2,heat_index_c__slot_2,loc_index__slot_3,air_temp_c__slot_3,rel_humidity_pct__slot_3,wind_speed_ms__slot_3,wind_dir_deg__slot_3,pressure_hpa__slot_3,precipitation_mm__slot_3,solar_wm2__slot_3,longwave_wm2__slot_3,surface_temp_c__slot_3,soil_moisture_pct__slot_3,ndvi__slot_3,pm25_ugm3__slot_3,pm10_ugm3__slot_3,co2_ppm__slot_3,no2_ppb__slot_3,o3_ppb__slot_3,noise_db__slot_3,traffic_index__slot_3,pedestrian_index__slot_3,sky_view_factor__slot_3,impervious_fraction__slot_3,water_proximity__slot_3,heat_index_c__slot_3,loc_index__slot_4,air_temp_c__slot_4,rel_humidity_pct__slot_4,wind_speed_ms__slot_4,wind_dir_deg__slot_4,pressure_hpa__slot_4,precipitation_mm__slot_4,solar_wm2__slot_4,longwave_wm2__slot_4,surface_temp_c__slot_4,soil_moisture_pct__slot_4,ndvi__slot_4,pm25_ugm3__slot_4,pm10_ugm3__slot_4,co2_ppm__slot_4,no2_ppb__slot_4,o3_ppb__slot_4,noise_db__slot_4,traffic_index__slot_4,pedestrian_index__slot_4,sky_view_factor__slot_4,impervious_fraction__slot_4,water_proximity__slot_4,heat_index_c__slot_4,loc_index__slot_1__d1,loc_index__slot_1__d2,loc_index__slot_1__d3,air_temp_c__slot_1__d1,air_temp_c__slot_1__d2,air_temp_c__slot_1__d3,rel_humidity_pct__slot_1__d1,rel_humidity_pct__slot_1__d2,rel_humidity_pct__slot_1__d3,wind_speed_ms__slot_1__d1,wind_speed_ms__slot_1__d2,wind_speed_ms__slot_1__d3,wind_dir_deg__slot_1__d1,wind_dir_deg__slot_1__d2,wind_dir_deg__slot_1__d3,pressure_hpa__slot_1__d1,pressure_hpa__slot_1__d2,pressure_hpa__slot_1__d3,precipitation_mm__slot_1__d1,precipitation_mm__slot_1__d2,precipitation_mm__slot_1__d3,solar_wm2__slot_1__d1,solar_wm2__slot_1__d2,solar_wm2__slot_1__d3,longwave_wm2__slot_1__d1,longwave_wm2__slot_1__d2,longwave_wm2__slot_1__d3,surface_temp_c__slot_1__d1,surface_temp_c__slot_1__d2,surface_temp_c__slot_1__d3,soil_moisture_pct__slot_1__d1,soil_moisture_pct__slot_1__d2,soil_moisture_pct__slot_1__d3,ndvi__slot_1__d1,ndvi__slot_1__d2,ndvi__slot_1__d3,pm25_ugm3__slot_1__d1,pm25_ugm3__slot_1__d2,pm25_ugm3__slot_1__d3,pm10_ugm3__slot_1__d1,pm10_ugm3__slot_1__d2,pm10_ugm3__slot_1__d3,co2_ppm__slot_1__d1,co2_ppm__slot_1__d2,co2_ppm__slot_1__d3,no2_ppb__slot_1__d1,no2_ppb__slot_1__d2,no2_ppb__slot_1__d3,o3_ppb__slot_1__d1,o3_ppb__slot_1__d2,o3_ppb__slot_1__d3,noise_db__slot_1__d1,noise_db__slot_1__d2,noise_db__slot_1__d3,traffic_index__slot_1__d1,traffic_index__slot_1__d2,traffic_index__slot_1__d3,pedestrian_index__slot_1__d1,pedestrian_index__slot_1__d2,pedestrian_index__slot_1__d3,sky_view_factor__slot_1__d1,sky_view_factor__slot_1__d2,sky_view_factor__slot_1__d3,impervious_fraction__slot_1__d1,impervious_fraction__slot_1__d2,impervious_fraction__slot_1__d3,water_proximity__slot_1__d1,water_proximity__slot_1__d2,water_proximity__slot_1__d3,heat_index_c__slot_1__d1,heat_index_c__slot_1__d2,heat_index_c__slot_1__d3,loc_index__slot_2__d1,loc_index__slot_2__d2,loc_index__slot_2__d3,air_temp_c__slot_2__d1,air_temp_c__slot_2__d2,air_temp_c__slot_2__d3,rel_humidity_pct__slot_2__d1,rel_humidity_pct__slot_2__d2,rel_humidity_pct_

# Alignment & Pipeline Notes

I added the alignment and pipeline explanatory note (also saved as `docs/slides/2026-05-29/alignment_pipeline_notes.md`).

---

Ah — this changes things again, and actually makes your setup much more coherent.

If:

```
loc_0 -> loc_1 -> loc_2
```

are consecutive points along a physical track/path, then the ordering IS meaningful. Even if locations correspond to near tree, beside building, shaded zone, open street, etc., that is actually GOOD.

Because environmental transitions along a path are exactly the kind of local structure shapelets can capture. So now your data is much closer to `multivariate spatial trajectories` rather than arbitrary spatial snapshots.

See `docs/slides/2026-05-29/alignment_pipeline_notes.md` for the full, verbatim note including recommended preprocessing and pipeline order (alignment -> normalization -> window/shapelet transforms).